# ASH-KV + SGLang: T4 Validation
This notebook is designed to test the ASH-KV INT8 Shadow Cache integration with SGLang on a T4 GPU (Colab/Kaggle). Since notebook cells run sequentially, we start the server as a background process.

In [ ]:
!git clone https://github.com/Ijas14/ASH-KV.git
%cd ASH-KV

# 1. Install lightweight SGLang dependencies manually to bypass pip's aggressive PyTorch/Triton downgrades
!pip install fastapi uvicorn outlines orjson pyzmq httpx pydantic numpy requests

# 2. Install SGLang core and router WITHOUT dependencies to protect Colab's PyTorch 2.11 / CUDA 12.8 environment
!pip install "sglang>=0.3.0" sglang-router --no-deps

# 3. Install ASH-KV
!pip install -e . --no-deps

# 4. Dynamically fetch the correct FlashInfer wheel for your futuristic PyTorch version
import torch
import os
cuda_ver = torch.version.cuda.replace('.', '')
torch_ver = '.'.join(torch.__version__.split('.')[:2])
url = f'https://flashinfer.ai/whl/cu{cuda_ver}/torch{torch_ver}/'
os.system(f'pip install flashinfer -i {url}')


In [ ]:
import torch
import triton
import sglang
import flashinfer

print(f"PyTorch Version: {torch.__version__}")
print(f"Triton Version: {triton.__version__}")
print(f"SGLang Version: {sglang.__version__}")
print(f"FlashInfer Version: {flashinfer.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
import subprocess
import time
import os

print("Starting SGLang server with ASH-KV...")
# Using a small 1.8B model to ensure it fits comfortably alongside the shadow cache on a 16GB T4
cmd = [
    "python", "-m", "sglang.launch_server",
    "--model-path", "Qwen/Qwen1.5-1.8B-Chat",
    "--mem-fraction-static", "0.4",  # Severely restrict memory to force early evictions
    "--port", "30000"
]
env = os.environ.copy()
env["ASHKV_ENABLED"] = "1"

server_process = subprocess.Popen(cmd, env=env, stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
print(f"Server started with PID: {server_process.pid}")
print("Waiting 60 seconds for server to initialize and load model weights...")
time.sleep(60)
print("Initialization wait complete. Check the logs in the next cell.")

In [ ]:
# Verify the server started successfully without crashing
!tail -n 50 server.log

In [ ]:
# STRESS TEST: Fire 32 concurrent requests at the server.
# Because we restricted memory to 0.4, this will trigger the SGLang eviction logic.
# The ASH-KV hooks will intercept these evictions, compress to INT8, and prevent OOM.
!python3 -m sglang.bench_serving --backend sglang --num-prompts 32 --port 30000

In [ ]:
# Check the logs for telemetry and circuit breaker stats after the run
!tail -n 100 server.log

In [ ]:
# Cleanup
server_process.terminate()
print("Server stopped.")